# 01 — Preprocessing
Load raw CSVs, clean data, run GLiNER NER (toponym masking), normalize text, and store results in DuckDB.

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import duckdb
from pathlib import Path

from reddit_np_topics.db import get_connection, init_schema, write_dataframe, table_row_count
from reddit_np_topics.preprocessing.cleaner import load_and_clean_all
from reddit_np_topics.preprocessing.ner import NERProcessor
from reddit_np_topics.preprocessing.normalizer import TextNormalizer

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Connect to DuckDB and set up schema
conn = get_connection(Path('..') / cfg['paths']['database'])
init_schema(conn)
print('DuckDB connected and schema ready.')

In [ ]:
# Step 1: Load and clean raw CSVs
raw_folder = Path('..') / cfg['paths']['raw_data']
df = load_and_clean_all(raw_folder)
write_dataframe(conn, df, 'raw_posts', mode='replace')
print(f'raw_posts: {table_row_count(conn, "raw_posts")} rows')

In [ ]:
# Step 2: NER — detect and mask location toponyms
ner_cfg = cfg['preprocessing']
ner = NERProcessor(
    model_name=ner_cfg['ner_model'],
    labels=ner_cfg['ner_labels'],
    threshold=ner_cfg['ner_threshold'],
)
df = ner.process_dataframe(df, text_col='text')
df.head(3)

In [ ]:
# Step 3: Text normalization
normalizer = TextNormalizer(extra_stopwords=ner_cfg['stopwords_extra'])
df['tokens'] = normalizer.normalize_series(df['tokens'])

# Drop rows with empty tokens after normalization
df = df.dropna(subset=['tokens'])
df = df[df['tokens'].str.strip() != '']
df.reset_index(drop=True, inplace=True)
print(f'{len(df)} documents after normalization')

In [ ]:
# Store normalized data to DuckDB
cols = ['origin_id','post_guid','park_name','user_guid','publish_date',
        'post_thumbnail_url','like_count','post_comment_count','post_url',
        'tags','emoji','post_title','body','text','tokens','loc_entities']
write_dataframe(conn, df[cols], 'normalized_posts', mode='replace')
print(f'normalized_posts: {table_row_count(conn, "normalized_posts")} rows')